In [ ]:
import yaml
import sys
from pathlib import Path

import pandas as pd
from collections import Counter

In [ ]:
# Load custom library and config
base_path = Path('../').resolve()
with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

sys.path.append(str(base_path))
from helpers import singlecell_utils

In [ ]:
print(cfg)

In [ ]:
h5ad_path = base_path / cfg['gene_h5ad']
dataset_id = cfg['dataset_id']
bam_dir = base_path / 'testdata/'

# Write barcode table

In [ ]:
ad = singlecell_utils.read_everything_but_X(h5ad_path)
df_obs = ad.obs

In [ ]:
df_obs.head().T

## Check if batch == bam file

In [ ]:
def batch_to_bam(s, bam_dir):
    f = s.split('-')
    return str(bam_dir) + '/' + '-'.join(f[:-1]) + '_' + f[-1] + '_L001.Aligned.sortedByCoord.out.bam'

In [ ]:
batch_list = list(df_obs['Batch'].drop_duplicates())

from_batch_bam_list = sorted([batch_to_bam(batch, bam_dir) for batch in batch_list])
from_glob_bam_list = sorted([str(p) for p in Path(bam_dir).glob('**/*.bam')])

len(set(from_batch_bam_list) & set(from_glob_bam_list)), len(set(from_batch_bam_list) - set(from_glob_bam_list)),  len(set(from_glob_bam_list) - set(from_batch_bam_list))

In [ ]:
set(from_batch_bam_list) - set(from_glob_bam_list)

In [ ]:
# Some batches are just not in the final h5ad
set(from_glob_bam_list) - set(from_batch_bam_list)

In [ ]:
print('How many Channel per donor?')
print(Counter(df_obs.groupby('individualID', observed=True)['Channel'].nunique()))
print()
print('How many Channel per individualID-Batch')
print(Counter(df_obs.groupby(['individualID', 'Batch'], observed=True)['Channel'].nunique()))
print()
print('Is libraryID != Channel ? ')
print(len(df_obs[df_obs['libraryID'].astype(str)!=df_obs['Channel'].astype(str)][['libraryID', 'Channel']]))

In [ ]:
cnt = df_obs.groupby('individualID', observed=True)['Channel'].nunique()
df_obs[df_obs['individualID'].isin(cnt[cnt > 1].index)][['individualID', 'libraryID', 'Channel', 'Batch']].drop_duplicates()

## Save minimal for snakemake

In [ ]:
check_cols = '''Batch
individualID
brain_region
Source
libraryID
Channel'''.split()

df_obs[check_cols].head(10)

In [ ]:
## df_obs['snRNA_bam'] =  df_obs['Batch'].apply(batch_to_bam, args=[bam_dir])
df_obs['cell_barcode'] = df_obs.index.str.split('_').map(lambda x: x[-1])
df_minimal = df_obs[check_cols + ['snRNA_bam', 'cell_barcode']]
df_minimal.head()

In [ ]:
len(set(df_minimal['individualID']))

In [ ]:
#df_minimal[df_minimal['Batch'].isin(df_minimal.head(10000)['Batch'].drop_duplicates())].to_parquet(f'./{dataset_id}_bam_path_and_barcodes.parquet')
df_minimal.to_parquet(f'./{dataset_id}_bam_path_and_barcodes.parquet')

## Check all bam exist

In [ ]:
#for col in ['snRNA_bam', 'ATAC_bam']:
for col in ['snRNA_bam']:
    unique_paths = df_minimal[col].unique()
    missing = [p for p in unique_paths if not Path(p).exists()]
    
    print(f"\n=== {col} ===")
    print(f"Total: {len(unique_paths)} | Missing: {len(missing)}")
    if missing:
        print("Missing files:")
        for p in missing:
            print(f"  {p}")

In [ ]:
print('How many Channel per bam?')
print(Counter(df_minimal.groupby(['snRNA_bam'], observed=True)['Channel'].nunique()))
print()